# CodeLens — Ingest `D:\test-codebase` into Neo4j

**Codebase structure:**
```
D:\test-codebase\
├── __init__.py
├── config.py
├── main.py
├── models/
│   ├── order.py
│   └── user.py
├── services/
│   ├── order_service.py
│   └── user_service.py
└── utils/
    └── helpers.py
```

**Run cells top to bottom. Each cell prints what it did.**

## Cell 1 — Install dependencies

In [5]:
%pip install neo4j --quiet
%pip install python-dotenv --quiet
print("Done")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Done



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2 — Config
Only thing you might need to change is `NEO4J_PASS` if you set a different password in Neo4j Desktop.

In [6]:
import ast
import os
from pathlib import Path
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()  



# ── Only change these if needed ──────────────────────────
CODEBASE_PATH = r"D:\test-codebase"
NEO4J_URI     = os.getenv("NEO4J_URI")
NEO4J_USER    = os.getenv("NEO4J_USER")
NEO4J_PASS    = os.getenv("NEO4J_PASS")
# ─────────────────────────────────────────────────────────

# Directories to skip when walking the codebase
SKIP_DIRS = {".venv", "venv", "__pycache__", ".git", 
             "migrations", "node_modules", "dist", "build"}

# Test Neo4j connection immediately
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
    driver.verify_connectivity()
    driver.close()
    print("Neo4j connection OK")
except Exception as e:
    print(f"Neo4j connection FAILED: {e}")
    print("Make sure your Neo4j Desktop DBMS is started")

Neo4j connection FAILED: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Make sure your Neo4j Desktop DBMS is started


## Cell 3 — Parser
Extracts functions, classes, imports from every `.py` file using Python's built-in `ast` module.

In [7]:
def make_node_id(file_path: str, class_name: str | None, func_name: str | None) -> str:
    """
    Stable unique ID for every node.
    Examples:
      services/user_service.py::UserService::get_user
      utils/helpers.py::format_date
      models/user.py::User
    """
    parts = [file_path]
    if class_name:
        parts.append(class_name)
    if func_name:
        parts.append(func_name)
    return "::".join(parts)


def extract_calls(func_node: ast.AST) -> list[str]:
    """All function/method calls inside a function body."""
    calls = []
    for node in ast.walk(func_node):
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                calls.append(node.func.id)
            elif isinstance(node.func, ast.Attribute):
                try:
                    obj  = ast.unparse(node.func.value)
                    attr = node.func.attr
                    calls.append(f"{obj}.{attr}")
                except Exception:
                    calls.append(node.func.attr)
    return list(set(calls))


def find_parent_class(tree: ast.Module, target_func: ast.AST) -> str | None:
    """Return class name if this function is a method, else None."""
    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
            # Direct children of this class body
            for child in node.body:
                if child is target_func:
                    return node.name
    return None


def parse_python_file(rel_path: str, source: str) -> dict:
    """
    Parse one .py file → dicts of functions, classes, imports.
    rel_path is relative to CODEBASE_PATH, forward-slashed.
    """
    try:
        tree = ast.parse(source)
    except SyntaxError as e:
        print(f"  [SKIP] {rel_path}: SyntaxError {e}")
        return {"functions": [], "classes": [], "imports": []}

    functions = []
    classes   = []
    imports   = []

    for node in ast.walk(tree):

        # ── Classes ───────────────────────────────────────────
        if isinstance(node, ast.ClassDef):
            bases = []
            for b in node.bases:
                if isinstance(b, ast.Name):
                    bases.append(b.id)
                elif isinstance(b, ast.Attribute):
                    try:
                        bases.append(ast.unparse(b))
                    except Exception:
                        pass

            classes.append({
                "id":         make_node_id(rel_path, node.name, None),
                "name":       node.name,
                "file":       rel_path,
                "start_line": node.lineno,
                "end_line":   node.end_lineno,
                "bases":      bases,
                "type":       "class",
            })

        # ── Functions & Methods ───────────────────────────────
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            class_name  = find_parent_class(tree, node)
            docstring   = ast.get_docstring(node)
            params      = [arg.arg for arg in node.args.args]
            return_type = ast.unparse(node.returns) if node.returns else None
            calls       = extract_calls(node)
            source_code = ast.get_source_segment(source, node) or ""

            functions.append({
                "id":          make_node_id(rel_path, class_name, node.name),
                "name":        node.name,
                "file":        rel_path,
                "class_name":  class_name,
                "start_line":  node.lineno,
                "end_line":    node.end_lineno,
                "docstring":   docstring,
                "params":      ", ".join(params),
                "return_type": return_type,
                "calls":       calls,
                "source":      source_code,
                "is_async":    isinstance(node, ast.AsyncFunctionDef),
                "type":        "function",
            })

        # ── Imports ───────────────────────────────────────────
        elif isinstance(node, ast.Import):
            for alias in node.names:
                imports.append({
                    "from_file": rel_path,
                    "module":    alias.name,
                    "names":     [],
                })

        elif isinstance(node, ast.ImportFrom):
            imports.append({
                "from_file": rel_path,
                "module":    node.module or "",
                "names":     [a.name for a in node.names],
            })

    return {"functions": functions, "classes": classes, "imports": imports}


print("Parser defined OK")

Parser defined OK


## Cell 4 — Parse all 8 files

In [8]:
root = Path(CODEBASE_PATH)

# Collect all .py files, skip noise dirs
py_files = [
    f for f in root.rglob("*.py")
    if not any(part in SKIP_DIRS for part in f.parts)
]

print(f"Files found: {len(py_files)}")
for f in sorted(py_files):
    print(f"  {f.relative_to(root)}")

# ── Parse every file ─────────────────────────────────────
all_functions = {}   # id → node dict
all_classes   = {}   # id → node dict
all_imports   = []   # list of import dicts

print("\nParsing...")
for file in sorted(py_files):
    rel_path = str(file.relative_to(root)).replace("\\", "/")
    source   = file.read_text(encoding="utf-8", errors="ignore")
    parsed   = parse_python_file(rel_path, source)

    fn_count  = len(parsed["functions"])
    cls_count = len(parsed["classes"])
    imp_count = len(parsed["imports"])

    for fn  in parsed["functions"]: all_functions[fn["id"]]  = fn
    for cls in parsed["classes"]:   all_classes[cls["id"]]   = cls
    all_imports.extend(parsed["imports"])

    print(f"  {rel_path:<40}  {fn_count} funcs  {cls_count} classes  {imp_count} imports")

print(f"""
── Totals ──────────────────────────
  Functions : {len(all_functions)}
  Classes   : {len(all_classes)}
  Imports   : {len(all_imports)}
────────────────────────────────────""")

Files found: 8
  __init__.py
  config.py
  main.py
  models\order.py
  models\user.py
  services\order_service.py
  services\user_service.py
  utils\helpers.py

Parsing...
  __init__.py                               0 funcs  0 classes  0 imports
  config.py                                 2 funcs  1 classes  2 imports
  main.py                                   1 funcs  0 classes  4 imports
  models/order.py                           3 funcs  1 classes  3 imports
  models/user.py                            5 funcs  2 classes  2 imports
  services/order_service.py                 2 funcs  1 classes  4 imports
  services/user_service.py                  4 funcs  1 classes  2 imports
  utils/helpers.py                          3 funcs  0 classes  1 imports

── Totals ──────────────────────────
  Functions : 20
  Classes   : 6
  Imports   : 18
────────────────────────────────────


## Cell 5 — Spot-check: print what was extracted
Make sure the parser found the right things before touching Neo4j.

In [9]:
print("=" * 55)
print("FUNCTIONS")
print("=" * 55)
for fn_id, fn in sorted(all_functions.items()):
    doc  = f"  doc: {fn['docstring'][:60]}..." if fn['docstring'] else ""
    ret  = f"  → {fn['return_type']}" if fn['return_type'] else ""
    asyn = " [async]" if fn['is_async'] else ""
    print(f"  {fn['file']}:{fn['start_line']:<4}  {fn['name']}({fn['params']}){ret}{asyn}")
    if doc:
        print(f"  {doc}")
    if fn['calls']:
        print(f"    calls: {fn['calls'][:6]}")

print()
print("=" * 55)
print("CLASSES")
print("=" * 55)
for cls_id, cls in sorted(all_classes.items()):
    bases = f"  extends: {cls['bases']}" if cls['bases'] else ""
    print(f"  {cls['file']}:{cls['start_line']:<4}  {cls['name']}{bases}")

print()
print("=" * 55)
print("IMPORTS")
print("=" * 55)
for imp in all_imports:
    names = f"  → {imp['names']}" if imp['names'] else ""
    print(f"  {imp['from_file']:<40}  {imp['module']}{names}")

FUNCTIONS
  config.py:13    get_settings(cls)  → Dict[str, Any]
  config.py:19    load_config()  → Config
    doc: Load and return configuration....
    calls: ['Config']
  main.py:8     main()
    doc: Run demo application....
    calls: ['calculate_total', 'bob.is_admin', 'OrderService', 'user_service.create_user', 'print', 'order_service.place_order']
  models/order.py:8     __init__(self, order_id, user, items, total)
  models/order.py:24    cancel(self)  → None
    doc: Cancel the order....
    calls: ['print']
  models/order.py:15    process_payment(self)  → bool
    doc: Simulate payment processing....
    calls: ['calculate_total', 'print']
  models/user.py:8     __init__(self, id)
    calls: ['datetime.now']
  models/user.py:12    save(self)  → bool
    doc: Simulate saving to database....
    calls: ['print']
  models/user.py:19    __init__(self, id, username, email, role)
    calls: ['super', 'super().__init__']
  models/user.py:30    add_order(self, order)  → None
    doc: 

## Cell 6 — Resolve calls → edges
Matches raw call strings like `"self.get_user"` to actual node IDs.

In [10]:
def resolve_calls(all_functions: dict) -> tuple[list, list]:
    """
    Returns:
      resolved_edges  → [{from, to}]             — internal CALLS edges
      external_edges  → [{from, raw_call}]        — CALLS_EXTERNAL edges
    """
    # Build name → [node_ids] index
    name_index = {}
    for node_id, fn in all_functions.items():
        name_index.setdefault(fn["name"], []).append(node_id)

    resolved = []
    external = []

    for node_id, fn in all_functions.items():
        for raw_call in fn["calls"]:
            # Strip self.x.y → take last segment
            call_name  = raw_call.split(".")[-1]
            candidates = name_index.get(call_name, [])

            if not candidates:
                external.append({"from": node_id, "raw_call": raw_call})
                continue

            # Prefer same-file match
            same_file = [c for c in candidates if fn["file"] in c]
            target    = same_file[0] if same_file else candidates[0]

            if target != node_id:   # skip self-loops
                resolved.append({"from": node_id, "to": target})

    # Deduplicate
    seen = set()
    unique_resolved = []
    for e in resolved:
        key = (e["from"], e["to"])
        if key not in seen:
            seen.add(key)
            unique_resolved.append(e)

    return unique_resolved, external


resolved_edges, external_edges = resolve_calls(all_functions)

print(f"Internal CALLS edges : {len(resolved_edges)}")
print(f"External CALLS edges : {len(external_edges)}")

print("\n── Internal edges (function → function) ──")
for e in resolved_edges:
    caller = e['from'].split("::")[-1]
    callee = e['to'].split("::")[-1]
    print(f"  {caller:<30} → {callee}")

print("\n── External edges (function → lib) ──")
for e in external_edges[:15]:   # show first 15
    caller = e['from'].split("::")[-1]
    print(f"  {caller:<30} → {e['raw_call']}")

Internal CALLS edges : 15
External CALLS edges : 21

── Internal edges (function → function) ──
  main                           → calculate_total
  main                           → is_admin
  main                           → create_user
  main                           → place_order
  main                           → load_config
  process_payment                → calculate_total
  __init__                       → __init__
  place_order                    → process_payment
  place_order                    → get_user
  place_order                    → add_order
  place_order                    → format_currency
  place_order                    → create_user
  create_user                    → validate_email
  create_user                    → save
  promote_to_admin               → get_user

── External edges (function → lib) ──
  load_config                    → Config
  main                           → OrderService
  main                           → print
  main                         

## Cell 7 — Write to Neo4j
Clears DB first so re-running is always clean. Then writes nodes → edges in the right order.

In [11]:
class Neo4jWriter:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def run(self, cypher, **params):
        with self.driver.session() as s:
            s.run(cypher, **params)

    def close(self):
        self.driver.close()

    # ── 0. Clear ──────────────────────────────────────────
    def clear(self):
        self.run("MATCH (n) DETACH DELETE n")
        print("  DB cleared")

    # ── 1. Indexes ────────────────────────────────────────
    def create_indexes(self):
        for cypher in [
            "CREATE INDEX fn_name  IF NOT EXISTS FOR (f:Function) ON (f.name)",
            "CREATE INDEX fn_file  IF NOT EXISTS FOR (f:Function) ON (f.file)",
            "CREATE INDEX cls_name IF NOT EXISTS FOR (c:Class)    ON (c.name)",
            "CREATE INDEX fi_path  IF NOT EXISTS FOR (f:File)     ON (f.path)",
        ]:
            self.run(cypher)
        print("  Indexes OK")

    # ── 2. File nodes ─────────────────────────────────────
    def write_files(self, all_functions, all_classes):
        files = set()
        for fn  in all_functions.values(): files.add(fn["file"])
        for cls in all_classes.values():   files.add(cls["file"])
        self.run(
            "UNWIND $files AS path MERGE (f:File {path: path})",
            files=list(files)
        )
        print(f"  File nodes     : {len(files)}")

    # ── 3. Class nodes + DEFINED_IN ───────────────────────
    def write_classes(self, all_classes):
        nodes = list(all_classes.values())
        self.run("""
            UNWIND $nodes AS node
            MERGE (c:Class {id: node.id})
            SET c.name       = node.name,
                c.file       = node.file,
                c.start_line = node.start_line,
                c.end_line   = node.end_line,
                c.bases      = node.bases
            WITH c, node
            MATCH (f:File {path: node.file})
            MERGE (c)-[:DEFINED_IN]->(f)
        """, nodes=nodes)
        print(f"  Class nodes    : {len(nodes)}")

    # ── 4. Function nodes + DEFINED_IN + BELONGS_TO ───────
    def write_functions(self, all_functions):
        nodes = list(all_functions.values())

        # Write function nodes + DEFINED_IN
        self.run("""
            UNWIND $nodes AS node
            MERGE (f:Function {id: node.id})
            SET f.name        = node.name,
                f.file        = node.file,
                f.class_name  = node.class_name,
                f.start_line  = node.start_line,
                f.end_line    = node.end_line,
                f.docstring   = node.docstring,
                f.params      = node.params,
                f.return_type = node.return_type,
                f.is_async    = node.is_async,
                f.source      = node.source
            WITH f, node
            MATCH (fi:File {path: node.file})
            MERGE (f)-[:DEFINED_IN]->(fi)
        """, nodes=nodes)

        # BELONGS_TO — methods only
        methods = [n for n in nodes if n.get("class_name")]
        if methods:
            self.run("""
                UNWIND $methods AS m
                MATCH (f:Function {id: m.id})
                MATCH (c:Class {name: m.class_name, file: m.file})
                MERGE (f)-[:BELONGS_TO]->(c)
            """, methods=methods)

        print(f"  Function nodes : {len(nodes)}  ({len(methods)} methods)")

    # ── 5. CALLS edges (internal) ─────────────────────────
    def write_calls(self, resolved_edges):
        if not resolved_edges:
            print("  CALLS edges    : 0")
            return
        self.run("""
            UNWIND $edges AS edge
            MATCH (a:Function {id: edge.from})
            MATCH (b:Function {id: edge.to})
            MERGE (a)-[:CALLS]->(b)
        """, edges=resolved_edges)
        print(f"  CALLS edges    : {len(resolved_edges)}")

    # ── 6. CALLS_EXTERNAL edges ───────────────────────────
    def write_external_calls(self, external_edges):
        if not external_edges:
            print("  EXTERNAL edges : 0")
            return
        self.run("""
            UNWIND $edges AS edge
            MERGE (lib:ExternalLib {name: edge.raw_call})
            WITH lib, edge
            MATCH (f:Function {id: edge.from})
            MERGE (f)-[:CALLS_EXTERNAL]->(lib)
        """, edges=external_edges)
        print(f"  EXTERNAL edges : {len(external_edges)}")

    # ── 7. INHERITS edges ─────────────────────────────────
    def write_inheritance(self, all_classes):
        name_index = {cls["name"]: cls["id"] for cls in all_classes.values()}
        inh_edges  = [
            {"child_id": cls["id"], "parent_id": name_index[base]}
            for cls in all_classes.values()
            for base in cls.get("bases", [])
            if base in name_index
        ]
        if inh_edges:
            self.run("""
                UNWIND $edges AS edge
                MATCH (child:Class  {id: edge.child_id})
                MATCH (parent:Class {id: edge.parent_id})
                MERGE (child)-[:INHERITS]->(parent)
            """, edges=inh_edges)
        print(f"  INHERITS edges : {len(inh_edges)}")

    # ── 8. IMPORTS edges (File → File for internal imports) 
    def write_imports(self, all_imports, all_files_set):
        """
        Create IMPORTS edges between files when one imports from another
        in the same codebase (e.g. from models.user import User).
        """
        import_edges = []
        # Build a lookup: module dotpath → file path
        # e.g. "models.user" → "models/user.py"
        module_to_file = {}
        for f in all_files_set:
            # models/user.py → models.user
            module_key = f.replace("/", ".").replace(".py", "")
            module_to_file[module_key] = f

        for imp in all_imports:
            target_file = module_to_file.get(imp["module"])
            if target_file and target_file != imp["from_file"]:
                import_edges.append({
                    "from_file": imp["from_file"],
                    "to_file":   target_file,
                    "names":     ", ".join(imp["names"]) if imp["names"] else "",
                })

        if import_edges:
            self.run("""
                UNWIND $edges AS edge
                MATCH (a:File {path: edge.from_file})
                MATCH (b:File {path: edge.to_file})
                MERGE (a)-[:IMPORTS {names: edge.names}]->(b)
            """, edges=import_edges)
        print(f"  IMPORTS edges  : {len(import_edges)}")


print("Neo4jWriter defined OK")

Neo4jWriter defined OK


## Cell 8 — Run the full write

In [12]:
all_file_paths = set()
for fn  in all_functions.values(): all_file_paths.add(fn["file"])
for cls in all_classes.values():   all_file_paths.add(cls["file"])

writer = Neo4jWriter(NEO4J_URI, NEO4J_USER, NEO4J_PASS)

print("Step 0 — Clear DB")
writer.clear()

print("\nStep 1 — Indexes")
writer.create_indexes()

print("\nStep 2 — Write nodes")
writer.write_files(all_functions, all_classes)
writer.write_classes(all_classes)
writer.write_functions(all_functions)

print("\nStep 3 — Write edges")
writer.write_calls(resolved_edges)
writer.write_external_calls(external_edges)
writer.write_inheritance(all_classes)
writer.write_imports(all_imports, all_file_paths)

writer.close()
print("\nAll done — open Neo4j Browser to explore the graph")

Step 0 — Clear DB


ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)

## Cell 9 — Verify: counts + summary

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

def q(cypher, **params):
    with driver.session() as s:
        return [dict(r) for r in s.run(cypher, **params)]

print("═" * 45)
print("NODE COUNTS")
print("═" * 45)
for row in q("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count DESC"):
    print(f"  {row['label']:<20} {row['count']}")

print()
print("═" * 45)
print("EDGE COUNTS")
print("═" * 45)
for row in q("MATCH ()-[r]->() RETURN type(r) AS rel, count(r) AS count ORDER BY count DESC"):
    print(f"  {row['rel']:<20} {row['count']}")

print()
print("═" * 45)
print("ALL FUNCTIONS")
print("═" * 45)
for row in q("MATCH (f:Function) RETURN f.file AS file, f.name AS name, f.start_line AS line, f.class_name AS cls ORDER BY file, line"):
    cls  = f"  [{row['cls']}]" if row['cls'] else ""
    print(f"  {row['file']}:{row['line']:<5} {row['name']}{cls}")

print()
print("═" * 45)
print("ALL CLASSES")
print("═" * 45)
for row in q("MATCH (c:Class) RETURN c.file AS file, c.name AS name, c.bases AS bases ORDER BY file"):
    bases = f"  extends {row['bases']}" if row['bases'] else ""
    print(f"  {row['file']}  {row['name']}{bases}")

print()
print("═" * 45)
print("INTERNAL CALL GRAPH")
print("═" * 45)
for row in q("MATCH (a:Function)-[:CALLS]->(b:Function) RETURN a.name AS caller, a.file AS caller_file, b.name AS callee, b.file AS callee_file ORDER BY caller"):
    print(f"  {row['caller']:<30} → {row['callee']}")
    print(f"  {'':30}   ({row['callee_file']})")

print()
print("═" * 45)
print("FILE IMPORT GRAPH")
print("═" * 45)
for row in q("MATCH (a:File)-[r:IMPORTS]->(b:File) RETURN a.path AS from_file, b.path AS to_file, r.names AS names"):
    print(f"  {row['from_file']:<35} imports {row['to_file']}")
    if row['names']:
        print(f"  {'':35}  → {row['names']}")

driver.close()

═════════════════════════════════════════════
NODE COUNTS
═════════════════════════════════════════════
  Function             20
  ExternalLib          14
  File                 7
  Class                6

═════════════════════════════════════════════
EDGE COUNTS
═════════════════════════════════════════════
  DEFINED_IN           26
  CALLS_EXTERNAL       21
  BELONGS_TO           15
  CALLS                15
  IMPORTS              12
  INHERITS             1

═════════════════════════════════════════════
ALL FUNCTIONS
═════════════════════════════════════════════
  config.py:13    get_settings  [Config]
  config.py:19    load_config
  main.py:8     main
  models/order.py:8     __init__  [Order]
  models/order.py:15    process_payment  [Order]
  models/order.py:24    cancel  [Order]
  models/user.py:8     __init__  [BaseModel]
  models/user.py:12    save  [BaseModel]
  models/user.py:19    __init__  [User]
  models/user.py:26    is_admin  [User]
  models/user.py:30    add_order  [Use

## Cell 10 — Useful Cypher queries to run in Neo4j Browser

Copy any of these into **Neo4j Browser** (`http://localhost:7474`) and hit run.
Switch to the **Graph** tab to see the visual.

---

### See the whole graph (all nodes + edges)
```cypher
MATCH (n)-[r]->(m) RETURN n, r, m
```

---

### See just the file import structure
```cypher
MATCH (a:File)-[r:IMPORTS]->(b:File)
RETURN a, r, b
```

---

### See a specific function and everything it calls
```cypher
MATCH path = (f:Function {name: "get_user"})-[:CALLS*1..3]->(dep)
RETURN path
```

---

### See who calls a specific function
```cypher
MATCH (caller:Function)-[:CALLS]->(f:Function {name: "get_user"})
RETURN caller.name, caller.file, caller.start_line
```

---

### See a class and all its methods
```cypher
MATCH (c:Class)<-[:BELONGS_TO]-(fn:Function)
RETURN c, fn
```

---

### See all services and what they call
```cypher
MATCH (f:Function)-[:CALLS]->(dep)
WHERE f.file CONTAINS "service"
RETURN f, dep
```

---

### Find all undocumented public functions
```cypher
MATCH (f:Function)
WHERE f.docstring IS NULL
  AND NOT f.name STARTS WITH '_'
RETURN f.file, f.name, f.start_line
ORDER BY f.file, f.start_line
```

---

### Full neighbourhood of any node (great for visual exploration)
```cypher
MATCH (n {name: "UserService"})-[r*1..2]-(neighbour)
RETURN n, r, neighbour
```